# E3.1 · Translating agentic risk upward

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E2.9 · Regulator and auditor conversations](https://spbreed.github.io/cyber-commons/lessons/E2.9.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Convert one blast-radius measurement into a board paragraph.

**Why a security engineer needs it.** Blast radius explained in engineering terms to a board that needs consequence. The control it builds is: what can happen, how fast, who can stop it.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The board does not want the threat model. It wants to know the exposure, whether it is going up or down, and what decision is being asked of them — in that order, in language that survives being repeated by someone else.

> **At CyberTravels.** The board does not want CyberTravels' threat model. It wants the exposure, the direction it is moving, and the decision being asked of them.

## 2 · The framework

```
   technical risk                board-usable exposure
   +-------------------+         +--------------------------+
   | prompt injection  |   -->   | exposure: X, trend: down |
   | in the RAG path   |         | decision asked: fund Y   |
   +-------------------+         +--------------------------+

   it has to survive being repeated by someone else, without you
```

Translating agentic risk upward means dropping every mechanism and keeping three
things: **exposure, likelihood, and the decision being requested.**

The failure mode is not using too much jargon. It is presenting *findings* when
the audience needs a *decision*. A board cannot act on "we found prompt
injection in the review agent". It can act on "a critical system can take N
units of unreviewed action, we demonstrated it, and we are asking for X or for
written acceptance".

That last clause matters more than people expect. **Accepting the risk in
writing, with a named owner and a review date, is a legitimate outcome.** Offering
it makes the ask credible, because it shows you are presenting a decision rather
than lobbying for a budget.

## 3 · The procedure, as a skill

The skill computes exposure, containment ASR and control coverage, prints the findings-shaped update with what is wrong with it, and then writes the position — same numbers, an ask attached.

In [ ]:
# skills/programme/risk-translation-upward/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: risk-translation-upward
description: >-
  Compute a fleet's exposure, containment effectiveness and control coverage,
  then translate the three numbers into a statement a board can act on rather
  than a list of findings. Use when preparing an executive update on agentic
  risk.
allowed-tools: Read, Grep, Glob
---

# A findings list is not a position

An executive update made of findings asks the reader to do the synthesis, and
they will do it wrong or not at all. Three computed numbers — how much the fleet
can reach, how much of an attack it stops, how much of the control set is
evidenced — support a position, a decision and a request.

## When to use this

Preparing a board or executive committee update, and whenever the current one is
a list of incidents.

## Procedure

**1 — Compute exposure.** Sum the blast radius across the fleet, weighted by
scope, with gated actions discounted. One number, and say what it is a sum of.

**2 — Compute containment effectiveness.** Attack success rate against the
controls as they run today, from the eval suite rather than from design intent.

**3 — Compute control coverage in three states.** Evidenced, stale, unevidenced.
The stale count is the one that moves an executive conversation, because it is
the one nobody expected.

**4 — Write the findings-shaped version and mark what is wrong with it.**
Usually: no trend, no comparison, no decision requested, no cost, and no
statement of what happens if nothing changes. Showing that is how the format
changes.

**5 — Write the position.** Where the exposure is concentrated, what would
reduce it most per unit of friction, what you are asking for, and what you will
report next time. Same numbers, different artefact.

## Output contract

```json
{
  "exposure": {"total": 0, "by_agent": [{"name": "str", "blast": 0, "gated": ["str"]}]},
  "containment": {"asr": 0.0, "from": "eval suite"},
  "coverage": {"evidenced": 0, "stale": 0, "unevidenced": 0, "of": 0},
  "findings_version": {"problems": ["str"]},
  "position": {"concentration": "str", "best_reduction": "str", "ask": "str", "next_report": "str"}
}
```

## Failure modes

- **Reporting incidents.** They are anecdotes at this altitude.
- **One coverage number.** The stale bucket is the story.
- **No ask.** An update with no decision requested gets noted.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/programme/risk-translation-upward/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/programme/risk-translation-upward/scripts/risk_translation_upward.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Compute fleet exposure, containment ASR and control coverage, and translate them into a board statement rather than a findings list.

This is the executable half of the `risk-translation-upward` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}

FLEET = {
 "pr-remediation-agent": [("read_file","self",True),
                          ("write_file","project",True),
                          ("deploy","org",False)],
 "claims-triage-agent":  [("read_file","self",True),
                          ("issue_refund","tenant",False)],
 "doc-summariser":       [("read_file","self",True)],
}
GATED = {"pr-remediation-agent": set(), "claims-triage-agent": {"issue_refund"},
         "doc-summariser": set()}

def blast(tools, gated):
    return sum(SCOPE_WEIGHT[s] * (1 if rev else 2)
               for n, s, rev in tools if n not in gated)

exposure = {a: blast(t, GATED[a]) for a, t in FLEET.items()}
print(f"{'agent':24s}{'blast radius':>14}")
print("-" * 40)
for a, b in sorted(exposure.items(), key=lambda kv: -kv[1]):
    print(f"{a:24s}{b:>14}")
total_exposure = sum(exposure.values())
print(f"{'FLEET TOTAL':24s}{total_exposure:>14}")

# likelihood — measured, from C1.2
ATTACKS = [("metadata service", False), ("path traversal", False),
           ("unlisted egress", True), ("denied tool", False)]
asr = sum(1 for _, through in ATTACKS if through) / len(ATTACKS)
print(f"\nred-team attack success rate (containment surface): {asr:.0%}")

# assurance — from E1.7
REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]
EVIDENCED = ["AC-1","AC-2","EV-1","EV-2"]
coverage = len(EVIDENCED) / len(REQUIRED)
print(f"controls currently evidenced: {len(EVIDENCED)}/{len(REQUIRED)} = {coverage:.0%}")

FINDINGS_UPDATE = """
This quarter the team identified prompt injection in the code review agent,
insufficient scope narrowing in the delegation chain, and gaps in our egress
allowlist. We ran garak and promptfoo against three agents and found a 25%
attack success rate on the containment surface. We recommend prioritising
provenance controls and completing the SPIFFE rollout.
"""
print(FINDINGS_UPDATE)
print("Problems with this, from the audience's side:")
for p in ["no exposure figure — how much can actually happen?",
          "'25% attack success rate' against what, and is that good or bad?",
          "four tool names nobody in the room can evaluate",
          "'recommend prioritising' is not a decision anyone can take",
          "no option to decline, so it reads as lobbying rather than a choice"]:
    print(f"   · {p}")

def board_translation(tier, exposure, asr, coverage, ask, cost, owner):
    likelihood = ("demonstrated" if asr > 0.2 else
                  "reduced but not eliminated" if asr > 0 else "not demonstrated")
    return f"""
EXPOSURE     A {tier}-tier system can take {exposure} units of unreviewed action.
             (One unit ≈ one irreversible change inside one project.)

LIKELIHOOD   We attacked it. {asr:.0%} of our attack suite succeeded — {likelihood}.
             This is a measurement, not an assessment.

ASSURANCE    {coverage:.0%} of the controls we say we operate are currently
             evidenced. The remainder are untested or their evidence has expired.

DECISION     {ask}
             Cost: {cost}.
             The alternative is to accept the unevidenced portion in writing,
             owned by {owner}, with a review date. Both are acceptable outcomes;
             we need one of them recorded."""

print(board_translation(
    tier="critical", exposure=total_exposure, asr=asr, coverage=coverage,
    ask="Fund continuous control verification for the agent fleet.",
    cost="0.5 FTE for two quarters, no new licences",
    owner="the Chief Operating Officer"))

# Verify: the translation must contain no mechanism and must offer a choice.
JARGON = ["prompt injection", "spiffe", "garak", "promptfoo", "cwe",
          "provenance", "allowlist", "delegation chain", "token exchange"]
text = board_translation("critical", total_exposure, asr, coverage,
                         "Fund continuous control verification.", "0.5 FTE", "the COO")
found = [j for j in JARGON if j in text.lower()]
print(f"mechanism terms present: {found or 'none'}")
has_choice = "alternative" in text.lower() and "accept" in text.lower()
has_number = str(total_exposure) in text and f"{asr:.0%}" in text
print(f"offers a genuine alternative : {has_choice}")
print(f"carries measured numbers     : {has_number}")
assert not found and has_choice and has_number
print("\nFour facts, no mechanism, and a decision that can go either way.")

## What you just proved

The fleet's exposure totals 46 units, containment ASR is 25%, and control coverage is 50%. The findings-shaped update is shown with five specific problems. The board translation states exposure, likelihood, assurance and a decision, contains no mechanism jargon, carries the measured numbers, and explicitly offers written acceptance as an alternative.

## Your turn

Write these four lines for your highest-tier system. If you cannot fill the likelihood line with a measurement, that is the first thing to fund — an assessment is not a number.

---

**Next → [E3.2 · Governing autonomy rather than approving tools](https://spbreed.github.io/cyber-commons/lessons/E3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*